# 3. Feature Engineering & Data Preparation

Create temporal features and prepare data for modeling using the refactored pipeline.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))
from src.pipeline import DemandForecastPipeline

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Initialize Pipeline and Load Data

In [ ]:
# Create pipeline for single store (for faster processing)
raw_path = '../data/raw'
store_filter = 'CA_1'

print(f"Loading and preparing data for {store_filter}...")
pipeline = DemandForecastPipeline(raw_path, split_date='2016-04-24', store_filter=store_filter)

# Load and prepare data (including feature engineering)
pipeline.load_and_prepare_data(optimize_memory=True)

print(f"\nData shape after feature engineering: {pipeline.df.shape}")
print(f"\nColumns created: {pipeline.df.columns.tolist()}")

## Inspect Features

In [ ]:
print("First rows of processed data:")
print(pipeline.df.head(10))

print("\nData types:")
print(pipeline.df.dtypes)

print("\nMissing values:")
missing = pipeline.df.isnull().sum()
print(missing[missing > 0])

## Calendar Features Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Day of month
axes[0, 0].hist(pipeline.df['day_of_month'].dropna(), bins=31, edgecolor='black')
axes[0, 0].set_xlabel('Day of Month')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Sales by Day of Month')

# Day of week
axes[0, 1].hist(pipeline.df['day_of_week'].dropna(), bins=7, edgecolor='black')
axes[0, 1].set_xlabel('Day of Week (0=Mon, 6=Sun)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Sales by Day of Week')

# Week of year
axes[1, 0].hist(pipeline.df['week_of_year'].dropna(), bins=52, edgecolor='black')
axes[1, 0].set_xlabel('Week of Year')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Sales by Week of Year')

# Sales distribution
axes[1, 1].hist(pipeline.df['sales'].dropna(), bins=50, edgecolor='black')
axes[1, 1].set_xlabel('Sales')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Sales Distribution')
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.show()

## Lagged Features Analysis

In [ ]:
# Select sample item for visualization
sample_item = pipeline.df['id'].unique()[0]
sample_data = pipeline.df[pipeline.df['id'] == sample_item].sort_values('date')

# Plot actual and lagged sales
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Actual sales
axes[0].plot(sample_data['date'], sample_data['sales'], marker='o', label='Sales')
axes[0].set_ylabel('Sales')
axes[0].set_title(f'Sales for {sample_item}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Lagged sales
axes[1].plot(sample_data['date'], sample_data['sales_lag_7'], marker='o', label='Lag-7', alpha=0.7)
axes[1].plot(sample_data['date'], sample_data['sales_lag_28'], marker='s', label='Lag-28', alpha=0.7)
axes[1].set_ylabel('Sales')
axes[1].set_title('Lagged Features')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Rolling means
axes[2].plot(sample_data['date'], sample_data['rolling_mean_7'], marker='o', label='Rolling Mean (7)', alpha=0.7)
axes[2].plot(sample_data['date'], sample_data['rolling_mean_28'], marker='s', label='Rolling Mean (28)', alpha=0.7)
axes[2].set_ylabel('Sales')
axes[2].set_title('Rolling Mean Features')
axes[2].set_xlabel('Date')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Sample item: {sample_item}")
print(f"Date range: {sample_data['date'].min()} to {sample_data['date'].max()}")

## Feature Correlation

In [ ]:
# Select numeric features for correlation
numeric_features = pipeline.df.select_dtypes(include=[np.number]).columns

# Calculate correlation with sales
correlations = pipeline.df[numeric_features].corr()['sales'].sort_values(ascending=False)

print("Feature Correlations with Sales:")
print(correlations)

# Plot top correlations
plt.figure(figsize=(10, 6))
correlations.head(15).plot(kind='barh')
plt.xlabel('Correlation')
plt.title('Top 15 Features Correlated with Sales')
plt.tight_layout()
plt.show()

## Train/Test Split

In [ ]:
# Prepare train/test split
pipeline.prepare_train_test()

print(f"Training set:")
print(f"  X_train shape: {pipeline.X_train.shape}")
print(f"  y_train shape: {pipeline.y_train.shape}")
print(f"  Target stats: mean={pipeline.y_train.mean():.2f}, std={pipeline.y_train.std():.2f}")

print(f"\nTest set:")
print(f"  X_test shape: {pipeline.X_test.shape}")
print(f"  Test samples: {len(pipeline.test_df)}")

print(f"\nFeature names ({len(pipeline.X_train.columns)} total):")
print(pipeline.X_train.columns.tolist())